<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">ספייק 2ב: <span dir="ltr">torch</span> ב-<span dir="ltr">Colab</span> על מעבד רגיל (<span dir="ltr">CPU</span>)</p>
<p dir="rtl" style="direction:rtl; text-align:right;">חלופה למחברת ה-<span dir="ltr">GPU</span>: התקנה מהירה של גרסת <span dir="ltr">CPU</span> מוכנה (כ-250 מגה-בייט, בלי הידור), ואימון קל שמתאים למעבד רגיל. סביבת ריצה: <b dir="ltr">Runtime → Change runtime type → R</b>, בלי <span dir="ltr">GPU</span>. הריצו תא אחרי תא. זמן כולל משוער: 5 עד 10 דקות.</p>


In [ ]:
# Timing helper: every step records its duration in a table printed at the end
timings <- data.frame(step = character(), seconds = numeric(), note = character())
tick <- function(step, expr, note = "") {
  t0 <- Sys.time()
  on.exit({
    s <- round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1)
    timings[nrow(timings) + 1, ] <<- list(step, s, note)
    cat(sprintf("[%s] %.1f s\n", step, s))
  })
  force(expr)
}
cat(R.version.string, "\n")


<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">1. התקנת <span dir="ltr">torch</span> לגרסת <span dir="ltr">CPU</span></p>
<p dir="rtl" style="direction:rtl; text-align:right;">החבילה מגיעה מוכנה מהשרת של <span dir="ltr">mlverse</span> וכוללת את ספריית <span dir="ltr">libtorch</span>, כך שאין הורדה נוספת. שאר החבילות מגיעות כקבצים בינאריים מוכנים מ-<span dir="ltr">Posit Package Manager</span>.</p>


In [ ]:
# Binary packages: dependencies from Posit Package Manager (Ubuntu jammy), torch from the mlverse CPU build
options(timeout = 900, HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
torch_version <- "0.17.0"   # current CRAN version (checked 5.11.26)
options(repos = c(
  torch = sprintf("https://torch-cdn.mlverse.org/packages/cpu/%s/", torch_version),
  CRAN  = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
tick("install torch (CPU binary)", install.packages("torch"))
tick("library(torch)", library(torch))
cat("torch version:", as.character(packageVersion("torch")), "
")
cat("torch ready:", torch_is_installed(), "
")


In [ ]:
tick("install luz + torchvision + helpers",
     install.packages(c("luz", "torchvision", "readr", "dplyr", "stringr", "jpeg", "tibble")))
library(luz); library(torchvision); library(dplyr); library(readr); library(stringr)
device <- "cpu"
x <- torch_randn(1000, 1000)
tick("1000x1000 matmul x20 on CPU", for (i in 1:20) y <- torch_mm(x, x), note = device)


<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">2. כיוונון <span dir="ltr">ResNet-18</span> על 72 תמונות של עזרי ניידות</p>
<p dir="rtl" style="direction:rtl; text-align:right;">הרשימה (<span dir="ltr">manifest</span>) נמצאת בקובץ <code dir="ltr">mobility_aids_manifest.csv</code> (בספר: <code dir="ltr">book_src/data/</code>). העלו אותו לסביבת העבודה (סמל התיקייה משמאל, העלאה), או הגדירו כתובת הורדה ב-<code dir="ltr">MANIFEST_URL</code>. התמונות מורדות מ-<span dir="ltr">Wikimedia Commons</span>.</p>


In [ ]:
# Get the manifest: upload it by hand in the Files pane, or set MANIFEST_URL (raw GitHub link)
MANIFEST_URL <- "https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/data/mobility_aids_manifest.csv"
if (!file.exists("mobility_aids_manifest.csv") && nzchar(MANIFEST_URL)) {
  download.file(MANIFEST_URL, "mobility_aids_manifest.csv")
}
stopifnot(file.exists("mobility_aids_manifest.csv"))
aids <- read_csv("mobility_aids_manifest.csv", show_col_types = FALSE) |>
  mutate(local = file.path("aids", sprintf("%s_%02d.jpg", class, row_number())), .by = class)
dir.create("aids", showWarnings = FALSE)
ua <- c("User-Agent" = "DSAI4PT-course-spike/0.1 (teaching use)")
tick("download 72 images", {
  for (i in seq_len(nrow(aids))) {
    if (!file.exists(aids$local[i])) {
      url <- paste0("https://commons.wikimedia.org/wiki/Special:FilePath/",
                    URLencode(str_replace_all(aids$file[i], " ", "_"), reserved = TRUE), "?width=320")
      try(download.file(url, aids$local[i], mode = "wb", headers = ua, quiet = TRUE))
      Sys.sleep(0.3)   # be polite to Wikimedia
    }
  }
})
ok <- file.exists(aids$local) & file.size(aids$local) > 1000
cat("downloaded:", sum(ok), "of", nrow(aids), "\n")
aids <- aids[ok, ]

In [ ]:
# Load every image as a 3x224x224 normalized tensor (ImageNet mean/std)
load_img <- function(path) {
  img <- jpeg::readJPEG(path)                       # height x width x channels in 0..1
  if (length(dim(img)) == 2) img <- array(rep(img, 3), c(dim(img), 3))   # gray -> RGB
  t <- torch_tensor(img[, , 1:3], dtype = torch_float())$permute(c(3, 1, 2))
  t <- nnf_interpolate(t$unsqueeze(1), size = c(224, 224), mode = "bilinear")$squeeze(1)
  m <- torch_tensor(c(0.485, 0.456, 0.406))$view(c(3, 1, 1))
  s <- torch_tensor(c(0.229, 0.224, 0.225))$view(c(3, 1, 1))
  (t - m) / s
}
set.seed(10)
aids <- aids |> mutate(split = if_else(row_number() %in% sample(n(), 8), "test", "train"), .by = class)
levels_ <- sort(unique(aids$class))
x_all <- torch_stack(lapply(aids$local, load_img))
y_all <- torch_tensor(match(aids$class, levels_), dtype = torch_long())
tr <- which(aids$split == "train"); te <- which(aids$split == "test")
dim(x_all)

<p dir="rtl" style="direction:rtl; text-align:right;">על מעבד רגיל מקפיאים את כל הרשת המאומנת מראש: מחשבים פעם אחת את המאפיינים (512 מספרים לכל תמונה) ומאמנים רק את שכבת הפלט. זה מהיר מאוד, וזו בדיוק הגישה של "מאפיינים מחושבים מראש".</p>


In [ ]:
# CPU-friendly transfer learning: frozen ResNet-18 as a feature extractor, train only a new output layer
torch_manual_seed(1)
tick("load pretrained ResNet-18", net <- model_resnet18(pretrained = TRUE))
net$fc <- nn_identity()   # drop the ImageNet output layer: the network now returns 512 features
net$eval()
tick("extract features of all images", feats <- with_no_grad(net(x_all)), note = "frozen backbone")
dim(feats)               # images x 512
head_img <- nn_linear(512, length(levels_))
opt <- optim_adam(head_img$parameters, lr = 0.01)
acc_hist <- c()
tick("train output layer 100 steps", {
  for (s in 1:100) {
    opt$zero_grad()
    loss <- nnf_cross_entropy(head_img(feats[tr, ]), y_all[tr])
    loss$backward(); opt$step()
    if (s %% 25 == 0) {
      pred <- with_no_grad(head_img(feats[te, ]))$argmax(dim = 2)
      acc_hist <- c(acc_hist, as.numeric((pred == y_all[te])$to(dtype = torch_float())$mean()))
      cat(sprintf("step %d  loss %.3f  test acc %.2f
", s, loss$item(), tail(acc_hist, 1)))
    }
  }
})


<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">3. משימת טקסט ל-<span dir="ltr">A5</span> עם הטבעות מחושבות מראש</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כל משפט קליני סינתטי כבר הומר להטבעה (<span dir="ltr">embedding</span>) של המודל <span dir="ltr">all-MiniLM-L6-v2</span>, והקובץ שמור במאגר הקורס. מאמנים עליו שכבת פלט קטנה, בלי צורך ב-<span dir="ltr">Python</span>.</p>


In [ ]:
# Pre-computed sentence embeddings (id, text, label, e1..e384) from the course repository
EMB_URL <- "https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/data/clinical_sentences_emb.csv"
tick("download embeddings", pre <- read_csv(EMB_URL, show_col_types = FALSE))
emb <- as.matrix(pre[, grep("^e[0-9]+$", names(pre))])
y <- match(pre$label, sort(unique(pre$label)))
set.seed(3)
te_t <- unlist(lapply(split(seq_along(y), y), function(i) sample(i, 2)))   # 2 test sentences per class
tr_t <- setdiff(seq_along(y), te_t)
X <- torch_tensor(emb, dtype = torch_float())
Y <- torch_tensor(y, dtype = torch_long())
torch_manual_seed(1)
head_txt <- nn_linear(ncol(emb), length(unique(y)))
opt <- optim_adam(head_txt$parameters, lr = 0.01)
tick("train text head 200 steps", {
  for (s in 1:200) {
    opt$zero_grad(); loss <- nnf_cross_entropy(head_txt(X[tr_t, ]), Y[tr_t]); loss$backward(); opt$step()
  }
})
pred <- head_txt(X[te_t, ])$argmax(dim = 2)
text_acc <- as.numeric((pred == Y[te_t])$to(dtype = torch_float())$mean())
cat("text test accuracy:", round(text_acc, 2), "on", length(te_t), "held-out sentences
")


<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">4. סיכום זמנים</p>
<p dir="rtl" style="direction:rtl; text-align:right;">העתיקו את הפלט של התא הבא ושלחו אותו. אם משהו נכשל, העתיקו גם את הודעת השגיאה.</p>


In [ ]:
print(timings)
cat("
TOTAL seconds:", sum(timings$seconds), "
")
cat("torch", as.character(packageVersion("torch")), "(CPU)",
    "| image acc:", paste(round(acc_hist, 2), collapse = ", "),
    "| text acc:", round(text_acc, 2), "
")
